# Image Processing Hardcore Project
Este notebook guia um projeto completo de processamento de imagens: pre-processamento avançado, detecção, extração de features, e treinamento de modelo.


In [ ]:
import os

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from skimage import exposure
from skimage.feature import hog, local_binary_pattern
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from image_processing import (
    adaptive_threshold,
    apply_gaussian_blur,
    canny_edge,
    closing,
    convert_to_gray,
    dilate,
    erode,
    extract_image_features,
    hit_or_miss,
    load_image,
    load_grayscale,
    opening,
    prewitt_edge,
    save_image,
    sobel_edge,
    laplacian_edge,
    threshold_binary,
)
from ml_pipeline import load_dataset_features, train_classifier, save_model

sns.set(style="whitegrid")


## Carregar e visualizar imagens

Neste notebook, vamos carregar imagens coloridas e exemplificar a visualização de múltiplas amostras. Isso facilita entender como cada transformação altera a imagem.


In [ ]:
sample_image = None
sample_path = "sample.jpg"

if os.path.exists(sample_path):
    sample_image = load_image(sample_path)
else:
    # Se não houver imagem local, crie uma imagem de exemplo usando gradiente BGR
    sample_image = np.zeros((256, 256, 3), dtype=np.uint8)
    sample_image[:, :, 0] = np.linspace(0, 255, 256, dtype=np.uint8)
    sample_image[:, :, 1] = np.linspace(255, 0, 256, dtype=np.uint8)[:, None]
    sample_image[:, :, 2] = 128

plt.figure(figsize=(8, 8))
plt.imshow(cv2.cvtColor(sample_image, cv2.COLOR_BGR2RGB))
plt.title("Imagem de exemplo colorida")
plt.axis("off")
plt.show()


## Pré-processamento avançado

A etapa de pré-processamento inclui normalização, equalização de histograma, redução de ruído e ajuste de contraste para deixar as imagens prontas para análise.


In [ ]:
gray = convert_to_gray(sample_image)

normalized = cv2.normalize(gray, None, 0, 255, cv2.NORM_MINMAX)
equalized = exposure.equalize_adapthist(gray, clip_limit=0.03)
blurred = apply_gaussian_blur(sample_image, ksize=11, sigma=3)

fig, axs = plt.subplots(1, 4, figsize=(18, 5))
axs[0].imshow(gray, cmap="gray")
axs[0].set_title("Escala de cinza")
axs[1].imshow(normalized, cmap="gray")
axs[1].set_title("Normalização")
axs[2].imshow(equalized, cmap="gray")
axs[2].set_title("Equalização")
axs[3].imshow(cv2.cvtColor(blurred, cv2.COLOR_BGR2RGB))
axs[3].set_title("Redução de ruído")
for ax in axs:
    ax.axis("off")
plt.show()


## Transformações geométricas e filtragem

Explore transformações geométricas como rotação, redimensionamento e transformação de perspectiva, além de filtros espaciais que acentuam bordas e texturas.


In [ ]:
height, width = sample_image.shape[:2]
center = (width // 2, height // 2)
rotation_matrix = cv2.getRotationMatrix2D(center, 30, 0.75)
rotated = cv2.warpAffine(sample_image, rotation_matrix, (width, height))
resized = cv2.resize(sample_image, (int(width * 0.6), int(height * 0.6)))

src_pts = np.float32([[0, 0], [width - 1, 0], [0, height - 1], [width - 1, height - 1]])
dst_pts = np.float32([[0, 0], [width - 1, 20], [20, height - 1], [width - 40, height - 1]])
warp_matrix = cv2.getPerspectiveTransform(src_pts, dst_pts)
warped = cv2.warpPerspective(sample_image, warp_matrix, (width, height))

sobel = sobel_edge(sample_image)
prewitt = prewitt_edge(sample_image)
laplacian = laplacian_edge(sample_image)

fig, axs = plt.subplots(2, 4, figsize=(20, 10))
axs[0, 0].imshow(cv2.cvtColor(sample_image, cv2.COLOR_BGR2RGB))
axs[0, 0].set_title("Original")
axs[0, 1].imshow(cv2.cvtColor(rotated, cv2.COLOR_BGR2RGB))
axs[0, 1].set_title("Rotação")
axs[0, 2].imshow(cv2.cvtColor(resized, cv2.COLOR_BGR2RGB))
axs[0, 2].set_title("Redimensionado")
axs[0, 3].imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
axs[0, 3].set_title("Perspectiva")

axs[1, 0].imshow(sobel, cmap="gray")
axs[1, 0].set_title("Sobel")
axs[1, 1].imshow(prewitt, cmap="gray")
axs[1, 1].set_title("Prewitt")
axs[1, 2].imshow(laplacian, cmap="gray")
axs[1, 2].set_title("Laplaciano")
axs[1, 3].imshow(cv2.cvtColor(blurred, cv2.COLOR_BGR2RGB))
axs[1, 3].set_title("Filtro Gaussiano")

for ax in axs.flatten():
    ax.axis("off")
plt.tight_layout()
plt.show()


## Detecção de bordas e segmentação

A detecção de bordas e a segmentação de imagem isolam estruturas relevantes para análise ou para alimentar modelos de machine learning.


In [ ]:
canny = canny_edge(sample_image)
ret, binary_mask = cv2.threshold(gray, 120, 255, cv2.THRESH_BINARY)

# Watershed segmentation com distância e máscara inicial
kernel = np.ones((3, 3), np.uint8)
opening_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel, iterations=2)
background = cv2.dilate(opening_mask, kernel, iterations=3)
foreground = cv2.erode(opening_mask, kernel, iterations=2)
dist_transform = cv2.distanceTransform(opening_mask, cv2.DIST_L2, 5)
ret, sure_fg = cv2.threshold(dist_transform, 0.6 * dist_transform.max(), 255, 0)
sure_fg = np.uint8(sure_fg)
unknown = cv2.subtract(background, sure_fg)

markers = cv2.connectedComponents(sure_fg)[1] + 1
markers[unknown == 255] = 0
markers = cv2.watershed(sample_image, markers)
segmented = sample_image.copy()
segmented[markers == -1] = [255, 0, 0]

fig, axs = plt.subplots(1, 3, figsize=(18, 6))
axs[0].imshow(canny, cmap="gray")
axs[0].set_title("Canny")
axs[1].imshow(binary_mask, cmap="gray")
axs[1].set_title("Máscara binária")
axs[2].imshow(cv2.cvtColor(segmented, cv2.COLOR_BGR2RGB))
axs[2].set_title("Watershed")
for ax in axs:
    ax.axis("off")
plt.show()


## Extração de características e análise

Extraímos descritores avançados como HOG, LBP e momentos regionais. Esses vetores podem ser usados como entradas para classificadores.


In [ ]:
hog_features, hog_image = hog(
    gray,
    orientations=9,
    pixels_per_cell=(16, 16),
    cells_per_block=(2, 2),
    visualize=True,
    multichannel=False,
)

lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")

moments = cv2.moments(binary_mask)
hu_moments = cv2.HuMoments(moments).flatten()

feature_record = extract_image_features(sample_image)
feature_record["hog_sum"] = float(np.sum(hog_features))
for i, value in enumerate(hu_moments):
    feature_record[f"hu_moment_{i}"] = float(value)

print("Feature vector sample:")
for key, value in list(feature_record.items())[:12]:
    print(f"{key}: {value}")

fig, axs = plt.subplots(1, 3, figsize=(18, 6))
axs[0].imshow(hog_image, cmap="gray")
axs[0].set_title("HOG")
axs[1].imshow(lbp, cmap="gray")
axs[1].set_title("LBP")
axs[2].bar(range(len(hu_moments)), hu_moments)
axs[2].set_title("Momentos de Hu")
for ax in axs:
    ax.axis("off")
plt.show()


## Treinar e avaliar modelo simples

Usamos um pipeline de scikit-learn para treinar um classificador RandomForest com validação cruzada.


In [ ]:
dataset_dir = "dataset"

if os.path.isdir(dataset_dir):
    df, labels = load_dataset_features(dataset_dir)
    print(f"Dataset encontrado com {len(df)} amostras e {len(df.columns)} features.")

    model, report = train_classifier(df, labels)
    print("Acurácia:", report["accuracy"])
    print(pd.DataFrame(report).transpose())

    cv_scores = cross_val_score(model, df, labels, cv=5)
    print("Cross-validation scores:", np.round(cv_scores, 3))
    print("Média:", np.round(cv_scores.mean(), 3))
else:
    print(f"Dataset não encontrado em '{dataset_dir}'. Coloque imagens rotuladas em subpastas para treino.")


## Exportar resultados e relatório visual

Salve os resultados processados e visualize os principais gráficos do projeto para entender quais transformações forneceram melhor desempenho.


In [ ]:
output_dir = "notebook_output"
os.makedirs(output_dir, exist_ok=True)

save_image(os.path.join(output_dir, "rotated.png"), rotated)
save_image(os.path.join(output_dir, "warped.png"), warped)
save_image(os.path.join(output_dir, "canny.png"), canny)

plt.figure(figsize=(10, 5))
plt.bar(["Sobel", "Prewitt", "Laplacian"], [np.mean(sobel), np.mean(prewitt), np.mean(laplacian)])
plt.title("Média de intensidade das bordas")
plt.ylabel("Valor médio de pixel")
plt.savefig(os.path.join(output_dir, "edge_strength.png"))
plt.show()

print(f"Imagens e métricas salvas em: {output_dir}")
